In [1]:
import anndata as ad
import os
import re
import numpy as np
import squidpy as sq
import scanpy as sc
import harmonypy as hm
import umap

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import linregress

In [ ]:
# Load preprocessed RNA matrices of 2 brain sections

adata_1st_brain = ad.read_h5ad('input/RNA_processed_1st_brain.h5ad')

In [ ]:
adata_2nd_brain = ad.read_h5ad('input/RNA_processed_2nd_brain.h5ad')

In [ ]:
common_vars = adata_1st_brain.var_names.intersection(adata_2nd_brain.var_names)
adata_1 = adata_1st_brain[:, common_vars].copy()
adata_2 = adata_2nd_brain[:, common_vars].copy()

In [ ]:
# combine 2 matrices into one

adata_both = ad.concat([adata_1, adata_2],
                      join="outer",     # keep all features
                      label="dataset",  # new column in .obs
                      keys=["1st_brain", "2nd_brain"])  

In [ ]:
common_genes = adata_1.var_names.intersection(adata_2.var_names)
len(common_genes)

## Make integrated UMAP of two brain sections using Harmony

In [ ]:
adata_both.layers["counts"] = adata_both.X.copy()
sc.pp.normalize_total(adata_both, inplace=True)
sc.pp.log1p(adata_both)

In [ ]:
sc.tl.pca(adata_both, svd_solver="arpack")

In [ ]:
ho = hm.run_harmony(adata_both.obsm['X_pca'], adata_both.obs, 'dataset', theta=4, lamb = 0.3, sigma = 0.02, nclust=30, max_iter_harmony = 10)   # 
# theta (diversity penalty) Default: 2 Larger values (e.g. 6, 8, 12) force Harmony to align datasets more strongly, discouraging dataset-specific structure.
# Larger (e.g. 50–100 for complex data): better representation of local structure, often making datasets spread more evenly.

# save Harmony-corrected embeddings
adata_both.obsm['X_pca_harmony'] = ho.Z_corr.T

In [ ]:
X_pca_1 = adata_both[adata_both.obs["dataset"]=="1st_brain"].obsm["X_pca_harmony"]
X_pca_2 = adata_both[adata_both.obs["dataset"]=="2nd_brain"].obsm["X_pca_harmony"]

In [ ]:

reducer = umap.UMAP(n_neighbors=5, min_dist=0.1, n_components=2, random_state=40)
reducer.fit(X_pca_1)
X_umap_1 = reducer.transform(X_pca_1)
X_umap_2 = reducer.transform(X_pca_2)

In [ ]:
adata_1.obsm["X_umap"] = X_umap_1
adata_2.obsm["X_umap"] = X_umap_2

adata_plot = ad.concat(
    [adata_1, adata_2],
    join="outer",
    label="dataset",
    keys=["1st brain section", "2nd brain section"],
    merge="unique"
)

adata_plot.obsm['X_pca_harmony'] = np.vstack([X_pca_1, X_pca_2])
adata_plot.obsm["X_umap"] = np.vstack([X_umap_1, X_umap_2])

sc.pl.umap(adata_plot, color=["dataset"], size=7, alpha=0.2)

In [ ]:
# UMAP is generated with randomness. Save matrix for downstream analysis

# adata_plot.write('harmony_RNA_2_brains.h5ad')

In [ ]:
# Load stored UMAP 
adata_plot = ad.read_h5ad('input/harmony_RNA_2_brains.h5ad')

In [ ]:
sc.set_figure_params(figsize=(15, 15))
sc.pl.umap(adata_plot, color=["dataset"], size=7, alpha=0.2)

In [ ]:
sc.set_figure_params(figsize=(15, 15))

ax = sc.pl.umap(
    adata_plot,
    color=["dataset"],
    size=4,
    alpha=0.9,
    show=False
)

# Rasterize only the scatter points
for coll in ax.collections:
    coll.set_rasterized(True)

# plt.savefig("RNA_harmony_umap.pdf", format="pdf", dpi=300, bbox_inches="tight")
plt.close()

## Cell clustering

In [ ]:
print("neighbors")
sc.pp.neighbors(adata_plot, n_neighbors=5, random_state=42, use_rep='X_pca_harmony')

In [ ]:
print("Leiden")
resolution = 2       # 1 -- 13 clusters
sc.tl.leiden(adata_plot, resolution=resolution, random_state=42)

In [ ]:
ad_1st = adata_plot[adata_plot.obs.dataset == '1st brain section']
ad_2nd = adata_plot[adata_plot.obs.dataset == '2nd brain section']

In [ ]:
sc.pl.umap(ad_1st, color=["leiden"], size=6)

In [ ]:
sq.pl.spatial_scatter(ad_1st, shape=None, color="leiden", size=6, library_id="one")

In [ ]:
sq.pl.spatial_scatter(ad_2nd, shape=None, color="leiden", size=6, library_id="one")

In [ ]:
# UMAP and clustering are random each time. Save clustering results.

# adata_plot.write('harmony_RNA_2_brains_cluster.h5ad')   
# ad_1st.write('harmony_RNA_Brain_2_cluster.h5ad')
# ad_2nd.write('harmony_RNA_Brain_1_cluster.h5ad')

In [ ]:
# Load stored clustering result

# adata_plot = ad.read_h5ad('RNA_UMAP_2_brains_clusters.h5ad') 
# ad_1st = ad.read_h5ad('harmony_RNA_Brain_1_cluster.h5ad') 